# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/khadeja-qureshi/Machine-Learning-Internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## Week-5 modeling goal

I remain in **Lane 2: Refresh / Content Opportunity Scoring**.

The task is to rank webpages by the observed likelihood that their average daily search impressions will decline by more than 20% after the decision moment.

The model uses only measurements from **March 1–15, 2026**. Measurements from **March 16–31, 2026** are used only to create the observed outcome label and evaluate the ranking.

June 2026 remains sealed and is not used for feature development, label design, or model selection.

I compare Logistic Regression and Random Forest with my Week-4 rule baseline. All methods use the same held-out clients, the same rows, and the same ranking metrics.

In [2]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise RuntimeError("HF_TOKEN is unavailable.")

print("HF_TOKEN loaded successfully.")

HF_TOKEN loaded successfully.


In [3]:
%pip install -q duckdb pandas numpy scikit-learn

In [4]:
import json
import os
import platform

import duckdb
import numpy as np
import pandas as pd
import sklearn

from IPython.display import Markdown, display
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
TOP_K_VALUES = [20, 50]

print("Libraries imported.")
print("Random seed:", RANDOM_STATE)

Libraries imported.
Random seed: 42


In [5]:
con = duckdb.connect()

safe_token = HF_TOKEN.replace("'", "''")

con.execute(f"""
    CREATE OR REPLACE SECRET hf_access (
        TYPE huggingface,
        TOKEN '{safe_token}'
    )
""")

REL = "hf://datasets/FlyRank/internship-warehouse"

MARCH_TABLE = f"""
    read_parquet(
        '{REL}/fact_content_daily_performance/month=2026-03/*.parquet'
    )
"""

print("Connected to the March 2026 warehouse partition.")

Connected to the March 2026 warehouse partition.


## Data verification

I use the March 2026 mid-panel partition. Before constructing the modeling frame, I verify the table's row count, date range, and expected daily grain.

The final June partition remains sealed.

In [6]:
warehouse_summary = con.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS min_report_date,
        MAX(report_date) AS max_report_date,
        COUNT(DISTINCT client_hash_id) AS client_count,
        COUNT(DISTINCT content_hash_id) AS content_count
    FROM {MARCH_TABLE}
""").df()

display(warehouse_summary)

assert str(
    warehouse_summary.loc[0, "min_report_date"]
)[:10] == "2026-03-01"

assert str(
    warehouse_summary.loc[0, "max_report_date"]
)[:10] == "2026-03-31"

print("March date-range check passed.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,row_count,min_report_date,max_report_date,client_count,content_count
0,9841378,2026-03-01,2026-03-31,55,331437


March date-range check passed.


In [7]:
grain_check = con.sql(f"""
    SELECT
        COUNT(*) AS duplicate_grain_groups
    FROM (
        SELECT
            report_date,
            client_hash_id,
            content_hash_id,
            COUNT(*) AS n
        FROM {MARCH_TABLE}
        GROUP BY
            report_date,
            client_hash_id,
            content_hash_id
        HAVING COUNT(*) > 1
    ) AS duplicate_groups
""").df()

display(grain_check)

assert int(
    grain_check.loc[0, "duplicate_grain_groups"]
) == 0

print(
    "Grain check passed: no duplicate "
    "date-client-content groups were found."
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,duplicate_grain_groups
0,0


Grain check passed: no duplicate date-client-content groups were found.


## Modeling frame

The warehouse source grain is one daily row per pseudonymized client and content item.

I aggregate those daily records into one row per client-content pair.

The feature window is March 1–15. The outcome window is March 16–31.

To reduce unstable measurements, I require:

- at least 10 observed feature-window days;
- at least 10 observed outcome-window days;
- at least 100 feature-window impressions;
- average search position between 1 and 20.

The position restriction matches the candidate population used by the Week-4 review rule.

In [8]:
page_frame = con.sql(f"""
    WITH page_windows AS (
        SELECT
            client_hash_id,
            content_hash_id,

            COUNT(DISTINCT report_date) FILTER (
                WHERE report_date BETWEEN DATE '2026-03-01'
                                      AND DATE '2026-03-15'
            ) AS active_days_first15,

            COUNT(DISTINCT report_date) FILTER (
                WHERE report_date BETWEEN DATE '2026-03-16'
                                      AND DATE '2026-03-31'
            ) AS outcome_days_present,

            SUM(gsc_impressions) FILTER (
                WHERE report_date BETWEEN DATE '2026-03-01'
                                      AND DATE '2026-03-15'
            ) AS impressions_first15,

            SUM(gsc_clicks) FILTER (
                WHERE report_date BETWEEN DATE '2026-03-01'
                                      AND DATE '2026-03-15'
            ) AS clicks_first15,

            SUM(
                CASE
                    WHEN report_date BETWEEN DATE '2026-03-01'
                                         AND DATE '2026-03-15'
                     AND gsc_impressions > 0
                    THEN gsc_avg_position * gsc_impressions
                    ELSE 0
                END
            )
            /
            NULLIF(
                SUM(gsc_impressions) FILTER (
                    WHERE report_date BETWEEN DATE '2026-03-01'
                                          AND DATE '2026-03-15'
                ),
                0
            ) AS avg_position_first15,

            SUM(gsc_impressions) FILTER (
                WHERE report_date BETWEEN DATE '2026-03-16'
                                      AND DATE '2026-03-31'
            ) AS impressions_outcome

        FROM {MARCH_TABLE}

        GROUP BY
            client_hash_id,
            content_hash_id
    )

    SELECT
        client_hash_id,
        content_hash_id,
        active_days_first15,
        outcome_days_present,
        impressions_first15,
        clicks_first15,

        100.0 * clicks_first15
            / NULLIF(impressions_first15, 0)
            AS ctr_pct_first15,

        avg_position_first15,

        impressions_first15
            / NULLIF(active_days_first15, 0)
            AS avg_daily_impressions_first15,

        impressions_outcome
            / NULLIF(outcome_days_present, 0)
            AS avg_daily_impressions_outcome

    FROM page_windows

    WHERE active_days_first15 >= 10
      AND outcome_days_present >= 10
      AND impressions_first15 >= 100
      AND avg_position_first15 BETWEEN 1 AND 20
""").df()

page_frame["is_declining"] = (
    page_frame["avg_daily_impressions_outcome"]
    < 0.80
    * page_frame["avg_daily_impressions_first15"]
).astype(int)

page_frame["actual_change_pct"] = (
    100
    * (
        page_frame["avg_daily_impressions_outcome"]
        - page_frame["avg_daily_impressions_first15"]
    )
    / page_frame["avg_daily_impressions_first15"]
)

page_frame["log_impressions_first15"] = np.log1p(
    page_frame["impressions_first15"]
)

page_frame["log_clicks_first15"] = np.log1p(
    page_frame["clicks_first15"]
)

print("Modeling rows:", f"{len(page_frame):,}")
print(
    "Clients:",
    page_frame["client_hash_id"].nunique(),
)
print(
    "Observed decline base rate:",
    round(page_frame["is_declining"].mean(), 3),
)

display(
    page_frame[
        [
            "active_days_first15",
            "impressions_first15",
            "clicks_first15",
            "ctr_pct_first15",
            "avg_position_first15",
            "is_declining",
        ]
    ].head()
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Modeling rows: 61,374
Clients: 36
Observed decline base rate: 0.313


,active_days_first15,impressions_first15,clicks_first15,ctr_pct_first15,avg_position_first15,is_declining
0,15,429.0,2.0,0.466200,4.386946,0
1,15,628.0,1.0,0.159236,5.265924,0
2,15,1280.0,9.0,0.703125,4.144531,0
3,15,2531.0,10.0,0.395101,4.861320,0
4,15,243.0,1.0,0.411523,14.534979,1


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

This is a supervised yes/no problem because the observed target records whether a page's average daily impressions declined by more than 20% after the decision moment.

The business output is a ranked review queue. I therefore use each classifier's predicted probability as a ranking score instead of treating its predicted class as an automatic decision.

I begin with **Logistic Regression** because it is simple, readable, and provides a useful test of whether a weighted combination of the honest signals can beat the Week-4 rule.

I also train a **Random Forest** as a nonlinear challenger. It can represent interactions such as CTR behaving differently at different search positions or visibility levels.

I do not prefer the Random Forest merely because it is more complex. The baseline and both models are evaluated using the same held-out clients and the same metrics.

The primary queue metrics are **Precision@20** and **Precision@50**. I also report Average Precision, ROC AUC, and lift over the observed validation base rate.

In [9]:
FEATURES = [
    "log_impressions_first15",
    "log_clicks_first15",
    "ctr_pct_first15",
    "avg_position_first15",
    "active_days_first15",
]

TARGET = "is_declining"
GROUP_COLUMN = "client_hash_id"

FORBIDDEN_FEATURE_TERMS = [
    "outcome",
    "future",
    "declin",
    "actual_change",
    "label",
    "trend",
    "client",
    "content_hash",
]

for feature in FEATURES:
    assert not any(
        forbidden in feature.lower()
        for forbidden in FORBIDDEN_FEATURE_TERMS
    ), f"Unsafe feature found: {feature}"

assert TARGET not in FEATURES
assert GROUP_COLUMN not in FEATURES
assert "content_hash_id" not in FEATURES

assert set(FEATURES).issubset(
    page_frame.columns
)

print("Honest model features:")
for feature in FEATURES:
    print("-", feature)

print("Target:", TARGET)
print("Group column:", GROUP_COLUMN)
print("Feature leakage check passed.")

Honest model features:
- log_impressions_first15
- log_clicks_first15
- ctr_pct_first15
- avg_position_first15
- active_days_first15
Target: is_declining
Group column: client_hash_id
Feature leakage check passed.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I use a grouped training-validation split based on `client_hash_id`.

Pages from a validation client are not allowed to appear in training. This is more honest than randomly splitting individual pages because pages from the same client may share site structure, content strategy, audience, and measurement patterns.

The Week-4 baseline, Logistic Regression, and Random Forest are evaluated on exactly the same validation rows.

The baseline's expected CTR values and impression reference are calculated from training clients only. Model preprocessing is also fitted on training rows only.

This is one development split from one mid-panel month. June 2026 remains sealed.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=RANDOM_STATE,
)

train_idx, validation_idx = next(
    splitter.split(
        X=page_frame[FEATURES],
        y=page_frame[TARGET],
        groups=page_frame[GROUP_COLUMN],
    )
)

train_frame = page_frame.iloc[
    train_idx
].copy()

validation_frame = page_frame.iloc[
    validation_idx
].copy()

train_clients = set(
    train_frame[GROUP_COLUMN]
)

validation_clients = set(
    validation_frame[GROUP_COLUMN]
)

assert train_clients.isdisjoint(
    validation_clients
)

assert train_frame[TARGET].nunique() == 2
assert validation_frame[TARGET].nunique() == 2

print("Training rows:", f"{len(train_frame):,}")
print(
    "Validation rows:",
    f"{len(validation_frame):,}",
)
print("Training clients:", len(train_clients))
print(
    "Validation clients:",
    len(validation_clients),
)
print(
    "Training base rate:",
    round(train_frame[TARGET].mean(), 3),
)
print(
    "Validation base rate:",
    round(validation_frame[TARGET].mean(), 3),
)
print("Client overlap:", len(
    train_clients.intersection(
        validation_clients
    )
))


Training rows: 39,619
Validation rows: 21,755
Training clients: 28
Validation clients: 8
Training base rate: 0.302
Validation base rate: 0.333
Client overlap: 0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

I reproduce the Week-4 rule inside this notebook so that the baseline and learned models are compared during the same run.

The baseline retains the Week-4 design:

- pages are compared with the median CTR of their search-position bucket;
- the score gives 70% weight to CTR deficit;
- the score gives 30% weight to impression-volume strength;
- pages without a measured CTR deficit rank after eligible pages.

The baseline's reference values are calculated using training clients only.

In [13]:
def add_position_bucket(frame):
    result = frame.copy()

    result["position_bucket"] = pd.cut(
        result["avg_position_first15"],
        bins=[
            -np.inf,
            3,
            10,
            20,
        ],
        labels=[
            "top_3",
            "page_1",
            "striking",
        ],
        right=True,
        include_lowest=True,
    )

    return result


train_baseline = add_position_bucket(
    train_frame
)

expected_ctr_by_bucket = (
    train_baseline
    .groupby(
        "position_bucket",
        observed=True,
    )["ctr_pct_first15"]
    .median()
)

global_train_ctr = (
    train_frame["ctr_pct_first15"]
    .median()
)

volume_reference = (
    train_frame["impressions_first15"]
    .quantile(0.95)
)

assert volume_reference > 0

print("Training-only expected CTR values:")
display(
    expected_ctr_by_bucket.to_frame(
        "expected_ctr_pct"
    )
)

print(
    "Training-only 95th percentile "
    "impression reference:",
    round(volume_reference, 2),
)

Training-only expected CTR values:


,expected_ctr_pct
position_bucket,
top_3,0.292326
page_1,0.220751
striking,0.113572


Training-only 95th percentile impression reference: 5857.1


In [14]:
def calculate_baseline_scores(
    frame,
    expected_ctr_lookup,
    volume_reference_value,
    fallback_ctr,
):
    scored = add_position_bucket(frame)

    expected_values = (
        scored["position_bucket"]
        .map(expected_ctr_lookup)
    )

    scored["expected_ctr_pct"] = (
        pd.to_numeric(
            expected_values,
            errors="coerce",
        )
        .fillna(fallback_ctr)
    )

    scored["ctr_deficit_ratio"] = (
        (
            scored["expected_ctr_pct"]
            - scored["ctr_pct_first15"]
        )
        /
        scored["expected_ctr_pct"].replace(
            0,
            np.nan,
        )
    ).clip(
        lower=0,
        upper=1,
    ).fillna(0)

    scored["volume_strength"] = (
        np.log1p(
            scored["impressions_first15"]
        )
        /
        np.log1p(
            volume_reference_value
        )
    ).clip(
        lower=0,
        upper=1,
    )

    scored["baseline_score"] = (
        100
        * (
            0.70
            * scored["ctr_deficit_ratio"]
            +
            0.30
            * scored["volume_strength"]
        )
    )

    # Preserve the Week-4 eligibility rule.
    scored.loc[
        scored["ctr_deficit_ratio"] <= 0,
        "baseline_score",
    ] = -1.0

    return scored


validation_baseline = (
    calculate_baseline_scores(
        frame=validation_frame,
        expected_ctr_lookup=(
            expected_ctr_by_bucket
        ),
        volume_reference_value=(
            volume_reference
        ),
        fallback_ctr=global_train_ctr,
    )
)

baseline_scores = (
    validation_baseline[
        "baseline_score"
    ].to_numpy()
)

print("Week-4 baseline scored.")
print(
    "Validation baseline score range:",
    round(float(np.min(baseline_scores)), 3),
    "to",
    round(float(np.max(baseline_scores)), 3),
)

Week-4 baseline scored.
Validation baseline score range: -1.0 to 100.0


In [15]:
X_train = train_frame[
    FEATURES
].copy()

y_train = train_frame[
    TARGET
].astype(int)

X_validation = validation_frame[
    FEATURES
].copy()

y_validation = validation_frame[
    TARGET
].astype(int)

print("Training matrix shape:", X_train.shape)
print(
    "Validation matrix shape:",
    X_validation.shape,
)

Training matrix shape: (39619, 5)
Validation matrix shape: (21755, 5)


In [16]:
logistic_model = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            ),
        ),
        (
            "scaler",
            StandardScaler(),
        ),
        (
            "model",
            LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

logistic_model.fit(
    X_train,
    y_train,
)

logistic_scores = (
    logistic_model.predict_proba(
        X_validation
    )[:, 1]
)

print("Logistic Regression trained.")

Logistic Regression trained.


In [17]:
random_forest_model = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            ),
        ),
        (
            "model",
            RandomForestClassifier(
                n_estimators=200,
                max_depth=8,
                min_samples_leaf=40,
                class_weight=(
                    "balanced_subsample"
                ),
                random_state=RANDOM_STATE,
                n_jobs=-1,
            ),
        ),
    ]
)

random_forest_model.fit(
    X_train,
    y_train,
)

random_forest_scores = (
    random_forest_model.predict_proba(
        X_validation
    )[:, 1]
)

print("Random Forest trained.")

Random Forest trained.


In [18]:
def top_k_indices(
    scores,
    tie_breaker,
    k,
):
    scores_array = np.asarray(
        scores,
        dtype=float,
    )

    tie_array = np.asarray(
        tie_breaker,
        dtype=float,
    )

    assert len(scores_array) == len(
        tie_array
    )

    order = np.lexsort(
        (
            -tie_array,
            -scores_array,
        )
    )

    return order[
        :min(k, len(order))
    ]


def precision_at_k(
    y_true,
    scores,
    tie_breaker,
    k,
):
    y_array = np.asarray(
        y_true,
        dtype=int,
    )

    selected = top_k_indices(
        scores=scores,
        tie_breaker=tie_breaker,
        k=k,
    )

    return float(
        y_array[selected].mean()
    )


def evaluate_method(
    method_name,
    y_true,
    scores,
    tie_breaker,
):
    y_array = np.asarray(
        y_true,
        dtype=int,
    )

    score_array = np.asarray(
        scores,
        dtype=float,
    )

    base_rate = float(
        y_array.mean()
    )

    precision_20 = precision_at_k(
        y_true=y_array,
        scores=score_array,
        tie_breaker=tie_breaker,
        k=20,
    )

    precision_50 = precision_at_k(
        y_true=y_array,
        scores=score_array,
        tie_breaker=tie_breaker,
        k=50,
    )

    return {
        "method": method_name,
        "validation_rows": int(
            len(y_array)
        ),
        "base_rate": base_rate,
        "correct_at_20": int(
            round(precision_20 * 20)
        ),
        "precision_at_20": (
            precision_20
        ),
        "lift_at_20": (
            precision_20 / base_rate
            if base_rate > 0
            else np.nan
        ),
        "correct_at_50": int(
            round(precision_50 * 50)
        ),
        "precision_at_50": (
            precision_50
        ),
        "lift_at_50": (
            precision_50 / base_rate
            if base_rate > 0
            else np.nan
        ),
        "average_precision": (
            average_precision_score(
                y_array,
                score_array,
            )
        ),
        "roc_auc": roc_auc_score(
            y_array,
            score_array,
        ),
    }

In [19]:
tie_breaker = (
    validation_frame[
        "impressions_first15"
    ].to_numpy()
)

comparison_table = pd.DataFrame(
    [
        evaluate_method(
            method_name=(
                "Week-4 baseline"
            ),
            y_true=y_validation,
            scores=baseline_scores,
            tie_breaker=tie_breaker,
        ),
        evaluate_method(
            method_name=(
                "Logistic Regression"
            ),
            y_true=y_validation,
            scores=logistic_scores,
            tie_breaker=tie_breaker,
        ),
        evaluate_method(
            method_name=(
                "Random Forest"
            ),
            y_true=y_validation,
            scores=random_forest_scores,
            tie_breaker=tie_breaker,
        ),
    ]
)

comparison_table = (
    comparison_table
    .sort_values(
        [
            "precision_at_50",
            "precision_at_20",
            "average_precision",
        ],
        ascending=False,
    )
    .reset_index(drop=True)
)

display(
    comparison_table.style.format(
        {
            "base_rate": "{:.3f}",
            "precision_at_20": "{:.3f}",
            "lift_at_20": "{:.2f}",
            "precision_at_50": "{:.3f}",
            "lift_at_50": "{:.2f}",
            "average_precision": "{:.3f}",
            "roc_auc": "{:.3f}",
        }
    )
)

,method,validation_rows,base_rate,correct_at_20,precision_at_20,lift_at_20,correct_at_50,precision_at_50,lift_at_50,average_precision,roc_auc
0,Random Forest,21755,0.333,13,0.650,1.95,28,0.560,1.68,0.445,0.640
1,Week-4 baseline,21755,0.333,12,0.600,1.80,28,0.560,1.68,0.412,0.602
2,Logistic Regression,21755,0.333,12,0.600,1.80,27,0.540,1.62,0.435,0.630


In [20]:
baseline_result = (
    comparison_table[
        comparison_table["method"]
        == "Week-4 baseline"
    ]
    .iloc[0]
)

winner_at_20 = (
    comparison_table
    .sort_values(
        [
            "precision_at_20",
            "average_precision",
        ],
        ascending=False,
    )
    .iloc[0]
)

winner_at_50 = (
    comparison_table
    .sort_values(
        [
            "precision_at_50",
            "average_precision",
        ],
        ascending=False,
    )
    .iloc[0]
)

validation_base_rate = float(
    validation_frame[TARGET].mean()
)

if (
    winner_at_20["method"]
    == winner_at_50["method"]
):
    queue_result_sentence = (
        f"**{winner_at_50['method']}** "
        "was strongest at both queue sizes."
    )
else:
    queue_result_sentence = (
        f"**{winner_at_20['method']}** "
        "was strongest at 20 pages, while "
        f"**{winner_at_50['method']}** "
        "was strongest at 50 pages. "
        "The preferred method therefore depends "
        "on the review capacity."
    )

comparison_markdown = f"""
### Observed model comparison

All three methods were evaluated on the same
**{len(validation_frame):,} validation pages**
from held-out clients.

At **Precision@20**, the strongest method was
**{winner_at_20['method']}**, with
**{winner_at_20['precision_at_20']:.3f}**
({int(winner_at_20['correct_at_20'])} correct pages
among the top 20).

The Week-4 baseline achieved
**{baseline_result['precision_at_20']:.3f}**
at the same queue size.

At **Precision@50**, the strongest method was
**{winner_at_50['method']}**, with
**{winner_at_50['precision_at_50']:.3f}**
({int(winner_at_50['correct_at_50'])} correct pages
among the top 50).

The Week-4 baseline achieved
**{baseline_result['precision_at_50']:.3f}**
at the same queue size.

The observed validation base rate was
**{validation_base_rate:.3f}**.

{queue_result_sentence}

These results are directional decision-support
evidence from one grouped development split.
They do not show that refreshing a selected page
would cause its search performance to recover.
"""

display(
    Markdown(comparison_markdown)
)


### Observed model comparison

All three methods were evaluated on the same
**21,755 validation pages**
from held-out clients.

At **Precision@20**, the strongest method was
**Random Forest**, with
**0.650**
(13 correct pages
among the top 20).

The Week-4 baseline achieved
**0.600**
at the same queue size.

At **Precision@50**, the strongest method was
**Random Forest**, with
**0.560**
(28 correct pages
among the top 50).

The Week-4 baseline achieved
**0.560**
at the same queue size.

The observed validation base rate was
**0.333**.

**Random Forest** was strongest at both queue sizes.

These results are directional decision-support
evidence from one grouped development split.
They do not show that refreshing a selected page
would cause its search performance to recover.


In [21]:
print("Random seed:", RANDOM_STATE)
print(
    "Python version:",
    platform.python_version(),
)
print(
    "pandas version:",
    pd.__version__,
)
print(
    "NumPy version:",
    np.__version__,
)
print(
    "DuckDB version:",
    duckdb.__version__,
)
print(
    "scikit-learn version:",
    sklearn.__version__,
)

Random seed: 42
Python version: 3.12.13
pandas version: 2.2.2
NumPy version: 2.0.2
DuckDB version: 1.3.2
scikit-learn version: 1.6.1


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

I interpret the strongest learned model rather than assuming that the most complex model is automatically best.

I use permutation importance to measure how much validation Average Precision falls when one feature is shuffled.

I then inspect:

- the top three features;
- false positives in the top-50 queue;
- declining pages that the model missed;
- three concrete wrong cases;
- whether errors are concentrated in particular search-position ranges.

Importance and error patterns describe this fitted model. They do not establish causal effects.

In [22]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
model_score_lookup = {
    "Logistic Regression": (
        logistic_scores
    ),
    "Random Forest": (
        random_forest_scores
    ),
}

trained_model_lookup = {
    "Logistic Regression": (
        logistic_model
    ),
    "Random Forest": (
        random_forest_model
    ),
}

learned_model_rows = (
    comparison_table[
        comparison_table["method"]
        != "Week-4 baseline"
    ]
    .copy()
)

best_model_name = (
    learned_model_rows
    .sort_values(
        [
            "precision_at_50",
            "precision_at_20",
            "average_precision",
        ],
        ascending=False,
    )
    .iloc[0]["method"]
)

best_model = trained_model_lookup[
    best_model_name
]

best_validation_scores = (
    model_score_lookup[
        best_model_name
    ]
)

print(
    "Learned model selected "
    "for interpretation:",
    best_model_name,
)


Learned model selected for interpretation: Random Forest


In [23]:
importance_result = (
    permutation_importance(
        estimator=best_model,
        X=X_validation,
        y=y_validation,
        scoring="average_precision",
        n_repeats=8,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
)

importance_table = pd.DataFrame(
    {
        "feature": FEATURES,
        "importance_mean": (
            importance_result
            .importances_mean
        ),
        "importance_std": (
            importance_result
            .importances_std
        ),
    }
).sort_values(
    "importance_mean",
    ascending=False,
)

display(
    importance_table.style.format(
        {
            "importance_mean": "{:.4f}",
            "importance_std": "{:.4f}",
        }
    )
)

print("Top three observed features:")
display(
    importance_table.head(3)
)

,feature,importance_mean,importance_std
2,ctr_pct_first15,0.0625,0.0023
0,log_impressions_first15,0.0203,0.0025
3,avg_position_first15,0.0185,0.0026
4,active_days_first15,0.0101,0.0008
1,log_clicks_first15,0.0098,0.0020


Top three observed features:


,feature,importance_mean,importance_std
2,ctr_pct_first15,0.062498,0.002252
0,log_impressions_first15,0.020325,0.002453
3,avg_position_first15,0.018496,0.002624


In [24]:
feature_explanations = {
    "log_impressions_first15": (
        "feature-window visibility may distinguish "
        "large, stable pages from smaller or more "
        "volatile pages"
    ),
    "log_clicks_first15": (
        "click activity reflects how much observed "
        "search demand converted into visits"
    ),
    "ctr_pct_first15": (
        "CTR measures observed click response relative "
        "to search-result exposure"
    ),
    "avg_position_first15": (
        "search position affects expected visibility "
        "and normal CTR behavior"
    ),
    "active_days_first15": (
        "coverage depth affects how stable the "
        "feature-window measurements are"
    ),
}

top_three_features = (
    importance_table
    .head(3)["feature"]
    .tolist()
)

feature_lines = []

for feature in top_three_features:
    feature_lines.append(
        f"- **`{feature}`**: "
        f"{feature_explanations[feature]}."
    )

importance_markdown = f"""
### Feature interpretation

Permutation importance indicates that
**{best_model_name}** relied most on the following
three features in this validation split:

{chr(10).join(feature_lines)}

These importance values measure how much the fitted
model's Average Precision changed when each feature
was shuffled.

They describe model reliance, not causal effects.
A feature appearing important does not mean that
changing that feature would cause future search
performance to improve.
"""

display(
    Markdown(importance_markdown)
)


### Feature interpretation

Permutation importance indicates that
**Random Forest** relied most on the following
three features in this validation split:

- **`ctr_pct_first15`**: CTR measures observed click response relative to search-result exposure.
- **`log_impressions_first15`**: feature-window visibility may distinguish large, stable pages from smaller or more volatile pages.
- **`avg_position_first15`**: search position affects expected visibility and normal CTR behavior.

These importance values measure how much the fitted
model's Average Precision changed when each feature
was shuffled.

They describe model reliance, not causal effects.
A feature appearing important does not mean that
changing that feature would cause future search
performance to improve.



### Feature interpretation

Permutation importance indicates that
**Random Forest** relied most on the following
three features in this validation split:

- **`ctr_pct_first15`**: CTR measures observed click response relative to search-result exposure.
- **`log_impressions_first15`**: feature-window visibility may distinguish large, stable pages from smaller or more volatile pages.
- **`avg_position_first15`**: search position affects expected visibility and normal CTR behavior.

These importance values measure how much the fitted
model's Average Precision changed when each feature
was shuffled.

They describe model reliance, not causal effects.
A feature appearing important does not mean that
changing that feature would cause future search
performance to improve.


In [26]:
full_ranking_order = top_k_indices(
    scores=best_validation_scores,
    tie_breaker=tie_breaker,
    k=len(validation_frame),
)

model_ranks = np.empty(
    len(validation_frame),
    dtype=int,
)

model_ranks[
    full_ranking_order
] = np.arange(
    1,
    len(validation_frame) + 1,
)

selected_top_50 = (
    model_ranks <= 50
)

error_frame = validation_frame[
    [
        "impressions_first15",
        "clicks_first15",
        "ctr_pct_first15",
        "avg_position_first15",
        "active_days_first15",
        "actual_change_pct",
        TARGET,
    ]
].copy()

error_frame["model_score"] = (
    best_validation_scores
)

error_frame["model_rank"] = (
    model_ranks
)

error_frame["selected_top_50"] = (
    selected_top_50
)

error_frame["error_group"] = np.select(
    [
        (
            error_frame["selected_top_50"]
            &
            (error_frame[TARGET] == 1)
        ),
        (
            error_frame["selected_top_50"]
            &
            (error_frame[TARGET] == 0)
        ),
        (
            ~error_frame["selected_top_50"]
            &
            (error_frame[TARGET] == 1)
        ),
    ],
    [
        "correct_top_50",
        "false_positive_top_50",
        "missed_decline",
    ],
    default="correct_not_selected",
)

error_summary = (
    error_frame
    .groupby("error_group")
    .agg(
        n=(TARGET, "size"),
        median_model_score=(
            "model_score",
            "median",
        ),
        median_impressions=(
            "impressions_first15",
            "median",
        ),
        median_clicks=(
            "clicks_first15",
            "median",
        ),
        median_ctr_pct=(
            "ctr_pct_first15",
            "median",
        ),
        median_position=(
            "avg_position_first15",
            "median",
        ),
        median_actual_change_pct=(
            "actual_change_pct",
            "median",
        ),
    )
    .round(3)
)

display(error_summary)

,n,median_model_score,median_impressions,median_clicks,median_ctr_pct,median_position,median_actual_change_pct
error_group,,,,,,,
correct_not_selected,14494,0.532,612.0,1.0,0.190,5.150,23.100
correct_top_50,28,0.708,7891.0,4.0,0.044,6.009,-50.786
false_positive_top_50,22,0.702,3946.5,1.0,0.041,2.523,3.164
missed_decline,7211,0.572,552.0,1.0,0.063,5.618,-46.539


In [27]:
error_frame["position_band"] = pd.cut(
    error_frame["avg_position_first15"],
    bins=[
        0,
        3,
        10,
        20,
    ],
    labels=[
        "top_3",
        "page_1",
        "striking",
    ],
    include_lowest=True,
)

error_by_position = (
    error_frame
    .groupby(
        "position_band",
        observed=True,
    )
    .agg(
        n=(TARGET, "size"),
        observed_decline_rate=(
            TARGET,
            "mean",
        ),
        selected_top_50_rate=(
            "selected_top_50",
            "mean",
        ),
        false_positive_count=(
            "error_group",
            lambda values: (
                values
                == "false_positive_top_50"
            ).sum(),
        ),
        missed_decline_count=(
            "error_group",
            lambda values: (
                values
                == "missed_decline"
            ).sum(),
        ),
    )
    .round(3)
)

display(error_by_position)

,n,observed_decline_rate,selected_top_50_rate,false_positive_count,missed_decline_count
position_band,,,,,
top_3,3951,0.281,0.006,12,1098
page_1,14195,0.358,0.002,10,5071
striking,3609,0.289,0.000,0,1042


In [28]:
false_positives = (
    error_frame[
        error_frame["error_group"]
        == "false_positive_top_50"
    ]
    .sort_values(
        [
            "model_score",
            "actual_change_pct",
        ],
        ascending=[
            False,
            False,
        ],
    )
)

missed_declines = (
    error_frame[
        error_frame["error_group"]
        == "missed_decline"
    ]
    .sort_values(
        [
            "actual_change_pct",
            "model_score",
        ],
        ascending=[
            True,
            False,
        ],
    )
)

wrong_cases = pd.concat(
    [
        false_positives.head(2),
        missed_declines.head(1),
    ]
)

if len(wrong_cases) < 3:
    remaining_wrong_cases = pd.concat(
        [
            false_positives.iloc[2:],
            missed_declines.iloc[1:],
        ]
    )

    remaining_wrong_cases = (
        remaining_wrong_cases.loc[
            ~remaining_wrong_cases.index.isin(
                wrong_cases.index
            )
        ]
    )

    wrong_cases = pd.concat(
        [
            wrong_cases,
            remaining_wrong_cases.head(
                3 - len(wrong_cases)
            ),
        ]
    )

wrong_cases = wrong_cases.head(3)

assert len(wrong_cases) == 3, (
    "Fewer than three wrong cases were available."
)

display(
    wrong_cases[
        [
            "error_group",
            "model_rank",
            "model_score",
            "impressions_first15",
            "clicks_first15",
            "ctr_pct_first15",
            "avg_position_first15",
            "actual_change_pct",
        ]
    ]
)

,error_group,model_rank,model_score,impressions_first15,clicks_first15,ctr_pct_first15,avg_position_first15,actual_change_pct
3821,false_positive_top_50,6,0.723835,8920.0,4.0,0.044843,6.809753,76.874299
19652,false_positive_top_50,7,0.721070,2358.0,1.0,0.042409,2.236217,-5.415076
33434,missed_decline,2290,0.601502,342.0,0.0,0.000000,5.497076,-100.000000


In [29]:
case_paragraphs = []

for case_number, (_, row) in enumerate(
    wrong_cases.iterrows(),
    start=1,
):
    near_threshold = (
        abs(
            row["actual_change_pct"] + 20
        )
        <= 5
    )

    if (
        row["error_group"]
        == "false_positive_top_50"
    ):
        explanation = (
            "The model ranked this page inside "
            "the top 50, but its observed later "
            "change did not cross the −20% decline "
            "threshold. Its visibility, clicks, CTR, "
            "or position resembled pages that did "
            "decline."
        )
    else:
        explanation = (
            "This page crossed the observed decline "
            "threshold but was not selected in the "
            "top 50. Its feature-window measurements "
            "did not resemble the patterns that the "
            "fitted model ranked most highly."
        )

    if near_threshold:
        threshold_sentence = (
            "The outcome was also close to the "
            "operational −20% boundary, so a small "
            "measurement difference could have "
            "changed the label."
        )
    else:
        threshold_sentence = (
            "The error cannot be explained only by "
            "being immediately next to the −20% "
            "label boundary."
        )

    case_text = f"""
### Wrong case {case_number}

- **Error type:** `{row['error_group']}`
- **Model rank:** {int(row['model_rank'])}
- **Model score:** {row['model_score']:.3f}
- **Feature-window impressions:** {row['impressions_first15']:,.0f}
- **Feature-window clicks:** {row['clicks_first15']:,.0f}
- **Feature-window CTR:** {row['ctr_pct_first15']:.3f}%
- **Average position:** {row['avg_position_first15']:.2f}
- **Observed later change:** {row['actual_change_pct']:.1f}%

{explanation}

{threshold_sentence}
"""

    case_paragraphs.append(case_text)

error_markdown = f"""
### Three concrete wrong cases

{chr(10).join(case_paragraphs)}

### Overall error interpretation

The five features do not measure query intent,
search-result features, content age, seasonality,
technical problems, competitor changes, or
client-specific events.

Some pages may therefore look similar in the
feature window but have different later outcomes.

The target is also an operational proxy. Pages at
−19.8% and −20.2% receive different labels even
though their measured changes are almost identical.
A classification error near that threshold may not
represent a completely wrong business judgment.
"""

display(
    Markdown(error_markdown)
)


### Three concrete wrong cases


### Wrong case 1

- **Error type:** `false_positive_top_50`
- **Model rank:** 6
- **Model score:** 0.724
- **Feature-window impressions:** 8,920
- **Feature-window clicks:** 4
- **Feature-window CTR:** 0.045%
- **Average position:** 6.81
- **Observed later change:** 76.9%

The model ranked this page inside the top 50, but its observed later change did not cross the −20% decline threshold. Its visibility, clicks, CTR, or position resembled pages that did decline.

The error cannot be explained only by being immediately next to the −20% label boundary.


### Wrong case 2

- **Error type:** `false_positive_top_50`
- **Model rank:** 7
- **Model score:** 0.721
- **Feature-window impressions:** 2,358
- **Feature-window clicks:** 1
- **Feature-window CTR:** 0.042%
- **Average position:** 2.24
- **Observed later change:** -5.4%

The model ranked this page inside the top 50, but its observed later change did not cross the −20% decline threshold. Its visibility, clicks, CTR, or position resembled pages that did decline.

The error cannot be explained only by being immediately next to the −20% label boundary.


### Wrong case 3

- **Error type:** `missed_decline`
- **Model rank:** 2290
- **Model score:** 0.602
- **Feature-window impressions:** 342
- **Feature-window clicks:** 0
- **Feature-window CTR:** 0.000%
- **Average position:** 5.50
- **Observed later change:** -100.0%

This page crossed the observed decline threshold but was not selected in the top 50. Its feature-window measurements did not resemble the patterns that the fitted model ranked most highly.

The error cannot be explained only by being immediately next to the −20% label boundary.


### Overall error interpretation

The five features do not measure query intent,
search-result features, content age, seasonality,
technical problems, competitor changes, or
client-specific events.

Some pages may therefore look similar in the
feature window but have different later outcomes.

The target is also an operational proxy. Pages at
−19.8% and −20.2% receive different labels even
though their measured changes are almost identical.
A classification error near that threshold may not
represent a completely wrong business judgment.


### Data limitation

The warehouse is an unbalanced client panel. Clients have different amounts of usable history and different measurement coverage.

This notebook uses one shared March window to remain comparable with the Week-4 baseline, but the result may be influenced by which clients had sufficient data during that month.

GA4 fields are deliberately excluded. Warehouse rows before a client's GA4 start date may contain zero-filled GA4 values with `ga4_data_available = FALSE`, and those zeros must not be interpreted as genuine zero engagement.

The observed decline target measures a short-window search-performance change. It does not prove that a page needs a refresh or that refreshing it would cause recovery.

In [30]:
os.makedirs(
    "work/outputs",
    exist_ok=True,
)

metrics_receipt = {
    "lane": (
        "refresh_content_opportunity_scoring"
    ),
    "feature_window": (
        "2026-03-01 to 2026-03-15"
    ),
    "outcome_window": (
        "2026-03-16 to 2026-03-31"
    ),
    "sealed_month_not_used": (
        "2026-06"
    ),
    "split_design": (
        "80/20 grouped holdout "
        "by client_hash_id"
    ),
    "random_state": RANDOM_STATE,
    "training_rows": int(
        len(train_frame)
    ),
    "validation_rows": int(
        len(validation_frame)
    ),
    "training_clients": int(
        len(train_clients)
    ),
    "validation_clients": int(
        len(validation_clients)
    ),
    "training_base_rate": float(
        train_frame[TARGET].mean()
    ),
    "validation_base_rate": float(
        validation_frame[TARGET].mean()
    ),
    "features": FEATURES,
    "selected_model_for_interpretation": (
        best_model_name
    ),
    "comparison_results": json.loads(
        comparison_table.to_json(
            orient="records"
        )
    ),
    "permutation_importance": json.loads(
        importance_table.to_json(
            orient="records"
        )
    ),
    "versions": {
        "python": (
            platform.python_version()
        ),
        "pandas": pd.__version__,
        "numpy": np.__version__,
        "duckdb": duckdb.__version__,
        "scikit_learn": (
            sklearn.__version__
        ),
    },
}

METRICS_PATH = (
    "work/outputs/"
    "w05_model_metrics.json"
)

with open(
    METRICS_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        metrics_receipt,
        file,
        indent=2,
    )

print(
    "Metrics receipt written to:",
    METRICS_PATH,
)

Metrics receipt written to: work/outputs/w05_model_metrics.json


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.